# Análise Financeira Empresarial — Superstore

Análise exploratória de dados de vendas de uma rede de lojas americana com **9.994 registros**.

**Variáveis analisadas:** Vendas, Lucro, Desconto, Categoria, Subcategoria, Segmento, Região, Estado e Clientes

## 1. Importações e Carregamento dos Dados

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

dados = pd.read_csv('Superstore.csv', encoding='latin-1')
dados['Order Date'] = pd.to_datetime(dados['Order Date'])
dados['Year']  = dados['Order Date'].dt.year
dados['Month'] = dados['Order Date'].dt.month
dados['Profit Margin'] = (dados['Profit'] / dados['Sales']) * 100

print('Shape:', dados.shape)
dados.head()

## 2. Vendas e Receita

In [ ]:
# Faturamento total por ano
faturamento_anual = dados.groupby('Year')['Sales'].sum()
print('--- Faturamento por Ano ---')
print(faturamento_anual)

In [ ]:
# Grafico de faturamento anual
ax = sns.lineplot(x='Year', y='Sales', data=dados.groupby('Year')['Sales'].sum().reset_index())
ax.figure.set_size_inches(12, 6)
plt.title('Faturamento Anual')
plt.xlabel('Ano')
plt.ylabel('Faturamento')
plt.show()

In [ ]:
# Faturamento por mês (sazonalidade)
faturamento_mensal = dados.groupby(['Year', 'Month'])['Sales'].sum()
faturamento_pivot  = faturamento_mensal.unstack(level=0)

sns.heatmap(faturamento_pivot, annot=True, fmt='.0f', cmap='YlGnBu')
plt.title('Faturamento por Mês e Ano')
plt.xlabel('Ano')
plt.ylabel('Mês')
plt.show()

In [ ]:
# Faturamento por categoria e subcategoria
faturamento_categoria    = dados.groupby('Category')['Sales'].sum().sort_values(ascending=False)
faturamento_subcategoria = dados.groupby('Sub-Category')['Sales'].sum().sort_values(ascending=False)
faturamento_regiao       = dados.groupby('Region')['Sales'].sum().sort_values(ascending=False)

print('--- Faturamento por Categoria ---')
print(faturamento_categoria)
print('\n--- Faturamento por Subcategoria ---')
print(faturamento_subcategoria)
print('\n--- Faturamento por Região ---')
print(faturamento_regiao)

## 3. Lucratividade

In [ ]:
# Lucro total por ano
lucro_anual = dados.groupby('Year')['Profit'].sum()
print('--- Lucro por Ano ---')
print(lucro_anual)

In [ ]:
# Margem de lucro por categoria
margem_categoria = dados.groupby('Category')['Profit Margin'].mean().sort_values()
print('--- Margem de Lucro por Categoria ---')
print(margem_categoria)

In [ ]:
# Grafico lucro por categoria
lucro_categoria = dados.groupby('Category')['Profit'].sum().sort_values(ascending=False).reset_index()
ax = sns.barplot(x='Category', y='Profit', data=lucro_categoria)
ax.figure.set_size_inches(12, 6)
plt.title('Lucro por Categoria')
plt.xlabel('Categoria')
plt.ylabel('Lucro')
plt.show()

## 4. Análise de Móveis (Furniture)

Furniture tem a menor margem de lucro (3.8%). Investigando as subcategorias para entender o motivo.

In [ ]:
furniture  = dados[dados['Category'] == 'Furniture']
margem_sub = furniture.groupby('Sub-Category')['Profit Margin'].mean().sort_values()
desconto_furniture = furniture.groupby('Sub-Category')['Discount'].mean().sort_values(ascending=False)

print('--- Margem por Subcategoria ---')
print(margem_sub)
print('\n--- Desconto médio por Subcategoria ---')
print(desconto_furniture)

In [ ]:
# Desconto vs Lucro em Furniture
ax = sns.scatterplot(x='Discount', y='Profit', data=furniture)
ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5, label='Ponto de equilíbrio')
ax.figure.set_size_inches(12, 6)
plt.title('Desconto vs Lucro para Móveis')
plt.xlabel('Desconto')
plt.ylabel('Lucro')
plt.legend()
plt.show()

## 5. Produtos

In [ ]:
# Produtos mais lucrativos e com maior prejuizo
produtos_lucrativos = dados.groupby('Product Name')['Profit'].sum().sort_values(ascending=False).head(10)
produtos_prejuizo   = dados.groupby('Product Name')['Profit'].sum().sort_values(ascending=True).head(10)

print('--- Top 10 Produtos Mais Lucrativos ---')
print(produtos_lucrativos)
print('\n--- Top 10 Produtos com Maior Prejuízo ---')
print(produtos_prejuizo)

## 6. Clientes

In [ ]:
# Top 10 clientes
clientes_top       = dados.groupby('Customer Name')['Sales'].count().sort_values(ascending=False).head(10)
clientes_lucrativos = dados.groupby('Customer Name')['Profit'].sum().sort_values(ascending=False).head(10)

print('--- Top 10 Clientes que Mais Compraram ---')
print(clientes_top)
print('\n--- Top 10 Clientes Mais Lucrativos ---')
print(clientes_lucrativos)

In [ ]:
# Segmento mais lucrativo
lucro_segmento = dados.groupby('Segment')['Profit'].sum().sort_values(ascending=False).reset_index()
ax = sns.barplot(x='Segment', y='Profit', data=lucro_segmento)
ax.figure.set_size_inches(12, 6)
plt.title('Lucro por Segmento')
plt.xlabel('Segmento')
plt.ylabel('Lucro')
plt.show()

## 7. Análise Regional

In [ ]:
regiao_faturamento = dados.groupby('Region')['Sales'].sum().sort_values(ascending=False)
regiao_lucro       = dados.groupby('Region')['Profit'].sum().sort_values(ascending=False)
estado_prejuizo    = dados.groupby('State')['Profit'].sum().sort_values(ascending=True).head(5)

print('--- Faturamento por Região ---')
print(regiao_faturamento)
print('\n--- Lucro por Região ---')
print(regiao_lucro)
print('\n--- Estados com Maior Prejuízo ---')
print(estado_prejuizo)

## 8. Mapa de Correlação

In [ ]:
correlacao = dados[['Sales', 'Profit', 'Discount', 'Quantity']].corr()
ax = sns.heatmap(correlacao, annot=True, cmap='coolwarm', fmt='.2f')
ax.figure.set_size_inches(10, 8)
plt.title('Mapa de Correlação entre Variáveis Numéricas')
plt.show()

## 9. Conclusões

- **Furniture** tem a menor margem de lucro (3.8%) — Tables (-14.8%) e Bookcases (-12.7%) vendem com prejuízo
- O motivo é o **desconto excessivo** — Tables com 26% e Bookcases com 21% de desconto médio
- A partir de **30% de desconto** praticamente toda venda vira prejuízo
- O desconto **não aumenta as vendas** (-0.028) mas **destrói o lucro** (-0.22)
- **Consumer** é o segmento mais lucrativo
- **Technology** e **Office Supplies** têm margens saudáveis acima de 13%